# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [5]:

class matrix:
    def __init__(self, n, m=None):
        if isinstance(n, list):
            if not n or not n[0]:
                raise ValueError("Matrix cannot be empty")
            if any(len(row) != len(n[0]) for row in n):
                raise ValueError("All rows must have the same length")
            self.data = [row[:] for row in n]
        else:
            if m is None or n <= 0 or m <= 0:
                raise ValueError("Invalid matrix dimensions")
            self.data = [[0 for j in range(m)] for i in range(n)]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(value, matrix):
            if len(self.data[key]) != len(value.data[0]):
                raise ValueError("Incompatible dimensions")
            self.data[key] = value.data[0][:]
        else:
            self.data[key] = value

    def assign(self, other):
        values = other.data if isinstance(other, matrix) else other
        if (len(values) != len(self.data) or
            any(len(a) != len(b) for a, b in zip(values, self.data))):
            raise ValueError("Matrix dimensions must match")
        self.data = [row[:] for row in values]

    def __repr__(self):
        return "\n".join(str(row) for row in self.data)


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [6]:

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        n, m = self.shape()
        return matrix([[self.data[i][j] for i in range(n)]
                       for j in range(m)])

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[row[n]] for row in self.data])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return matrix([row[n_0:n_1]
                       for row in self.data[m_0:m_1]])


In [7]:
 def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            if isinstance(i, slice) or isinstance(j, slice):
                rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
                return matrix([row[j] if isinstance(j, slice)
                               else [row[j]] for row in rows])
            return self.data[i][j]
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [8]:
def constant(n, m, c):
    return matrix([[float(c) for j in range(m)] for i in range(n)])
def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    M = zeros(n, n)
    for i in range(n):
        M[i, i] = 1.0
    return M


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [9]:
    def scalarmul(self, c):
        return matrix([[x * c for x in row] for row in self.data])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match")
        return matrix([[self[i, j] + N[i, j]
                        for j in range(self.shape()[1])]
                       for i in range(self.shape()[0])])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match")
        return matrix([[self[i, j] - N[i, j]
                        for j in range(self.shape()[1])]
                       for i in range(self.shape()[0])])

    def mat_mult(self, N):
        n, m = self.shape()
        p, q = N.shape()
        if m != p:
            raise ValueError("Incompatible dimensions")
        return matrix([[sum(self[i, k] * N[k, j] for k in range(m))
                        for j in range(q)] for i in range(n)])

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match")
        return matrix([[self[i, j] * N[i, j]
                        for j in range(self.shape()[1])]
                       for i in range(self.shape()[0])])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


In [10]:
def __mul__(self, other):
    if isinstance(other, matrix):
        return self.mat_mult(other)
    return self.scalarmul(other)

def __rmul__(self, other):
    return self.scalarmul(other)

def __add__(self, other):
    return self.add(other)

def __sub__(self, other):
    return self.sub(other)

def __eq__(self, other):
    return self.equals(other)

6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [11]:

A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
C = A
I = eye(2)

print("1. Associative Property: (AB)C = A(BC)")
print((A * B) * C == A * (B * C))

print("\n2. Distributive Property: A(B+C) = AB + AC")
print(A * (B + C) == A * B + A * C)

print("\n3. Matrix Multiplication is Not Commutative: AB != BA")
print(A * B != B * A)

print("\n4. Identity Property: AI = A")
print(A * I == A)


1. Associative Property: (AB)C = A(BC)


TypeError: unsupported operand type(s) for *: 'matrix' and 'matrix'